# NYC Airbnb Market Analysis — EngiViz DataViz Hackathon
**Track:** Python (Matplotlib, Seaborn, Plotly)

**Story:** NYC's Airbnb market looks saturated, but a third of it is a mirage — dormant listings and commercial operators inflate the numbers, while price and actual guest demand barely correlate.


## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
PALETTE = {"Manhattan": "#6C5CE7", "Brooklyn": "#00B894", "Queens": "#FDCB6E",
           "Bronx": "#E17055", "Staten Island": "#0984E3"}

df = pd.read_csv('AB_NYC_2019.csv')  # adjust path for your environment
df.head()


## 2. Clean the Data
Remove $0 listings (data errors) and cap price for readable plots (raw values kept for stats).

In [ ]:
df_clean = df[df['price'] > 0].copy()
df_clean['price_capped'] = df_clean['price'].clip(upper=500)

print(f"Rows total: {len(df)} | after removing $0 listings: {len(df_clean)}")
print(f"Listings never available (availability_365==0): {(df['availability_365']==0).sum()} "
      f"({(df['availability_365']==0).mean()*100:.1f}%)")


## 3. Chart 1 — Where listings are, colored by borough, sized by demand

In [ ]:
fig, ax = plt.subplots(figsize=(11, 10))
for grp, color in PALETTE.items():
    sub = df_clean[df_clean['neighbourhood_group'] == grp]
    ax.scatter(sub['longitude'], sub['latitude'],
               s=np.clip(sub['number_of_reviews'], 3, 200) * 0.6,
               c=color, alpha=0.35, label=grp, edgecolors='none')
ax.set_title("NYC Airbnb Listings — Location, Borough & Demand\n(bubble size = number of reviews)",
             fontsize=16, fontweight='bold')
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
leg = ax.legend(title="Borough", loc='upper left', markerscale=1.5, fontsize=11)
for lh in leg.legend_handles:
    lh.set_alpha(1); lh.set_sizes([80])
plt.tight_layout()
plt.savefig('chart1_geomap.png', dpi=150)
plt.show()


## 4. Chart 2 — Price distribution by borough & room type

In [ ]:
order = df_clean.groupby('neighbourhood_group')['price'].median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(12, 7))
sns.boxplot(data=df_clean, x='neighbourhood_group', y='price_capped', hue='room_type',
            order=order, showfliers=False, ax=ax,
            palette={"Entire home/apt": "#6C5CE7", "Private room": "#00B894", "Shared room": "#FDCB6E"})
ax.set_title("Price Distribution by Borough & Room Type\n(prices capped at $500 for readability)",
             fontsize=16, fontweight='bold')
ax.set_xlabel("Borough"); ax.set_ylabel("Price per night ($)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
plt.legend(title="Room Type", loc='upper right')
plt.tight_layout()
plt.savefig('chart2_price_by_borough.png', dpi=150)
plt.show()


## 5. Chart 3 — Does higher price mean more demand?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
sample = df_clean.sample(min(8000, len(df_clean)), random_state=42)
ax.scatter(sample['price'], sample['number_of_reviews'],
           c=sample['neighbourhood_group'].map(PALETTE), alpha=0.35, s=25, edgecolors='none')
ax.set_xscale('log')
ax.set_title("Price vs. Number of Reviews\n(higher price does not mean more demand)",
             fontsize=16, fontweight='bold')
ax.set_xlabel("Price per night ($, log scale)"); ax.set_ylabel("Number of reviews")
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=10, label=b)
           for b, c in PALETTE.items()]
ax.legend(handles=handles, title="Borough", loc='upper right')
plt.tight_layout()
plt.savefig('chart3_price_vs_reviews.png', dpi=150)
plt.show()

corr = df_clean[['price', 'number_of_reviews']].corr().iloc[0, 1]
print(f"Correlation price vs number_of_reviews: {corr:.3f}")


## 6. Chart 4 — Market concentration among 'power hosts'

In [ ]:
host_counts = (df_clean.groupby(['host_id', 'host_name'])['id'].count()
               .reset_index(name='listing_count')
               .sort_values('listing_count', ascending=False).head(15))
host_counts['label'] = host_counts['host_name'].fillna('Unknown') + " (" + host_counts['host_id'].astype(str) + ")"

fig, ax = plt.subplots(figsize=(11, 8))
bars = ax.barh(host_counts['label'][::-1], host_counts['listing_count'][::-1], color="#6C5CE7")
ax.set_title("Top 15 Hosts by Number of Listings\n(market concentration among 'power hosts')",
             fontsize=16, fontweight='bold')
ax.set_xlabel("Number of listings")
for bar in bars:
    w = bar.get_width()
    ax.text(w + 3, bar.get_y() + bar.get_height()/2, f"{int(w)}", va='center', fontsize=10)
plt.tight_layout()
plt.savefig('chart4_top_hosts.png', dpi=150)
plt.show()

top_host_share = host_counts['listing_count'].sum() / len(df_clean) * 100
print(f"Top 15 hosts alone control {top_host_share:.1f}% of all listings")
print("Note: #1 (Sonder) and #2 (Blueground) are commercial property-management companies, not individuals.")


## 7. Chart 5 — How much 'listed' inventory is actually dormant?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
ax.hist(df_clean['availability_365'], bins=40, color="#00B894", edgecolor='white', alpha=0.85)
never_avail = (df_clean['availability_365'] == 0).sum()
pct = never_avail / len(df_clean) * 100
ax.axvline(0, color='#E17055', linestyle='--', linewidth=2)
ax.annotate(f"{never_avail:,} listings ({pct:.1f}%)\nnever available all year",
            xy=(5, ax.get_ylim()[1]*0.85), fontsize=13, color='#E17055', fontweight='bold')
ax.set_title("Listing Availability Over the Year\n(a large share of 'active' listings are effectively dormant)",
             fontsize=16, fontweight='bold')
ax.set_xlabel("Days available out of 365"); ax.set_ylabel("Number of listings")
plt.tight_layout()
plt.savefig('chart5_availability.png', dpi=150)
plt.show()


## 8. Bonus — Interactive Plotly Map (hover tooltips)
Run this cell in an environment with Plotly installed (e.g. Kaggle notebooks have it pre-installed). This produces an interactive HTML map — great for the 'creative dashboard design' criterion.

In [ ]:
import plotly.express as px

fig = px.scatter_mapbox(
    df_clean.sample(min(10000, len(df_clean)), random_state=42),
    lat="latitude", lon="longitude",
    color="neighbourhood_group",
    size=np.clip(df_clean['number_of_reviews'], 3, 200),
    hover_name="name",
    hover_data={"price": True, "room_type": True, "number_of_reviews": True,
                "latitude": False, "longitude": False},
    color_discrete_map=PALETTE,
    zoom=10, height=700,
    title="NYC Airbnb Listings — Interactive Map (hover for details)"
)
fig.update_layout(mapbox_style="carto-positron", margin={"r":0,"t":40,"l":0,"b":0})
fig.write_html("interactive_map.html")
fig.show()


## 9. Combined Dashboard (single export image)

In [ ]:
fig = plt.figure(figsize=(22, 13))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.28)
fig.suptitle("NYC Airbnb Market — Where the Listed Inventory Isn't What It Looks Like",
             fontsize=22, fontweight='bold', y=0.99)

ax1 = fig.add_subplot(gs[0, 0])
for grp, color in PALETTE.items():
    sub = df_clean[df_clean['neighbourhood_group'] == grp]
    ax1.scatter(sub['longitude'], sub['latitude'],
                s=np.clip(sub['number_of_reviews'], 3, 200) * 0.4,
                c=color, alpha=0.3, label=grp, edgecolors='none')
ax1.set_title("Listings by Borough & Demand", fontsize=14, fontweight='bold')
ax1.set_xlabel("Longitude", fontsize=10); ax1.set_ylabel("Latitude", fontsize=10)
ax1.legend(fontsize=8, loc='upper left', markerscale=0.8)
ax1.tick_params(labelsize=9)

ax2 = fig.add_subplot(gs[0, 1])
sns.boxplot(data=df_clean, x='neighbourhood_group', y='price_capped', hue='room_type',
            order=order, showfliers=False, ax=ax2,
            palette={"Entire home/apt": "#6C5CE7", "Private room": "#00B894", "Shared room": "#FDCB6E"})
ax2.set_title("Price by Borough & Room Type", fontsize=14, fontweight='bold')
ax2.set_xlabel(""); ax2.set_ylabel("Price ($)", fontsize=10)
ax2.yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
ax2.tick_params(axis='x', rotation=20, labelsize=9)
ax2.legend(fontsize=7, title_fontsize=8)

ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(sample['price'], sample['number_of_reviews'],
            c=sample['neighbourhood_group'].map(PALETTE), alpha=0.3, s=15, edgecolors='none')
ax3.set_xscale('log')
ax3.set_title(f"Price vs. Reviews (corr = {corr:.2f})", fontsize=14, fontweight='bold')
ax3.set_xlabel("Price ($, log)", fontsize=10); ax3.set_ylabel("Reviews", fontsize=10)
ax3.tick_params(labelsize=9)

ax4 = fig.add_subplot(gs[1, 0])
ax4.barh(host_counts['host_name'].fillna('Unknown').head(10)[::-1],
         host_counts['listing_count'].head(10)[::-1], color="#6C5CE7")
ax4.set_title("Top 10 Hosts by Listings", fontsize=14, fontweight='bold')
ax4.set_xlabel("Listings", fontsize=10)
ax4.tick_params(labelsize=9)

ax5 = fig.add_subplot(gs[1, 1])
ax5.hist(df_clean['availability_365'], bins=40, color="#00B894", edgecolor='white', alpha=0.85)
ax5.axvline(0, color='#E17055', linestyle='--', linewidth=2)
ax5.set_title(f"Availability — {pct:.0f}% Never Available", fontsize=14, fontweight='bold')
ax5.set_xlabel("Days available / 365", fontsize=10); ax5.set_ylabel("Listings", fontsize=10)
ax5.tick_params(labelsize=9)

ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
kpi_text = (
    f"KEY FINDINGS\n\n"
    f"Total listings analyzed:  {len(df_clean):,}\n\n"
    f"Median price/night:  ${df_clean['price'].median():.0f}\n\n"
    f"Listings never available all year:\n   {never_avail:,}  ({pct:.1f}%)\n\n"
    f"Price <-> demand correlation:\n   {corr:.3f}  (essentially none)\n\n"
    f"Top host by volume:\n   {host_counts.iloc[0]['host_name']}  "
    f"({host_counts.iloc[0]['listing_count']} listings)\n\n"
    f"Manhattan share of listings:\n   "
    f"{(df_clean['neighbourhood_group']=='Manhattan').mean()*100:.1f}%"
)
ax6.text(0.02, 0.98, kpi_text, transform=ax6.transAxes, fontsize=13,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.6', facecolor='#F5F3FF', edgecolor='#6C5CE7', linewidth=1.5))

plt.savefig('airbnb_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Summary

- **35.9%** of listings (17,530) are available **0 days/year** — a huge share of "active" inventory is dormant.
- Price and guest demand are **essentially uncorrelated** (r = -0.048) — higher price does not buy more bookings.
- Manhattan commands a clear price premium (median ~$190 for entire homes) but isn't where demand is most efficient.
- Top hosts are increasingly **commercial operators** (Sonder, Blueground), not individuals renting a spare room.
- **Conclusion:** NYC's Airbnb market looks saturated, but a meaningful share of it is a mirage — dormant listings and commercial operators inflate the numbers, while price and actual guest demand barely correlate.